# 10.3 Grid Search

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.03_grid-search.ipynb)

Companion code for Section 10.3 of the book. Run the cells in order from top to bottom.

**Example 10.3.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
import time
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target

# ── Grid Search ───────────────────────────────────────────────────
param_grid_rf = {
    'n_estimators':    [50, 100, 200],
    'max_depth':       [None, 5, 10, 20],
    'max_features':    ['sqrt', 'log2', 0.5],
    'min_samples_leaf':[1, 2, 5],
}

n_combinations = 1
for vals in param_grid_rf.values():
    n_combinations *= len(vals)
print(f'Grid Search: {n_combinations} combinations × 5-fold = {n_combinations*5} fits')

t0 = time.time()
gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=0,
    return_train_score=True
)
gs.fit(X, y)
t_grid = time.time() - t0

print(f'Grid Search completed in {t_grid:.1f}s')
print(f'Best params: {gs.best_params_}')
print(f'Best CV AUC: {gs.best_score_:.4f}')

# ── Visualise: heatmap of AUC vs n_estimators × max_depth ────────
import pandas as pd
cv_results = pd.DataFrame(gs.cv_results_)

# Pivot for heatmap (fixing other params at best)
pivot_data = cv_results.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators',
    aggfunc='max'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Grid Search Analysis - Random Forest', fontsize=14, fontweight='bold')

ax = axes[0]
import seaborn as sns
sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlOrRd',
           ax=ax, linewidths=0.5, vmin=0.985, vmax=1.0)
ax.set_title('AUC Heatmap: n_estimators × max_depth\n(max over other params)')

ax = axes[1]
# All scores sorted
all_scores = cv_results['mean_test_score'].sort_values(ascending=False)
ax.plot(range(len(all_scores)), all_scores.values, 'b-', lw=1.5, alpha=0.7)
ax.axhline(gs.best_score_, color='red', ls='--', lw=2,
          label=f'Best: {gs.best_score_:.4f}')
ax.fill_between(range(len(all_scores)), all_scores.values,
               gs.best_score_*0.995, alpha=0.1, color='green', label='Top 0.5%')
ax.set_xlabel('Configuration (ranked)'); ax.set_ylabel('CV AUC')
ax.set_title('All Grid Configurations Ranked')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('grid_search.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Grid search report ────────────────────────────────────────────
print('\nTop 5 configurations:')
top5 = cv_results.nlargest(5, 'mean_test_score')[
    ['param_n_estimators','param_max_depth','param_max_features',
     'param_min_samples_leaf','mean_test_score','std_test_score']
]
print(top5.to_string(index=False))